# iTAPAS デモ: Sioux Falls ネットワーク

iTAPAS (incremental Traffic Assignment by Paired Alternative Segments) による静的利用者均衡配分の実行例。

**参考文献:**
- Bar-Gera, H. (2010). Traffic assignment by paired alternative segments. *Transportation Research Part B*, 44(8-9), 1022-1046.
- Xie, J. & Xie, C. (2016). New insights and improvements of using paired alternative segments for traffic assignment. *Transportation Research Part B*, 93, 406-424.

## 1. データの読み込み

GMNS Plus フォーマットの Sioux Falls ネットワーク (24ノード, 76リンク, 528 OD対) を読み込む。

In [ ]:
from network import Network
from cost_function import bpr_cost, bpr_cost_derivative, bpr_cost_integral
from itapas import ITAPAS

# GMNS Plus 形式のネットワークデータ読み込み
data_dir = "data/SiouxFalls"
net = Network(f"{data_dir}/node.csv", f"{data_dir}/link.csv", f"{data_dir}/demand.csv")

print(f"ノード数: {net.num_nodes}")
print(f"リンク数: {net.num_links}")
print(f"起点数:   {len(net.origins)}")
print(f"OD対数:   {len(net.demand)}")
print(f"総需要:   {sum(net.demand.values()):.0f}")

## 2. iTAPAS による均衡配分

BPR関数をリンクコスト関数として使用し、iTAPASで利用者均衡配分を実行する。

$$t_a(f_a) = t_a^0 \left(1 + \alpha \left(\frac{f_a}{c_a}\right)^\beta \right)$$

In [ ]:
solver = ITAPAS(net, bpr_cost, bpr_cost_derivative, bpr_cost_integral)

link_flow, log = solver.solve(
    max_iter=50,
    gap_threshold=1e-3,
    inner_iterations=10,
    mu=1e-5,
    verbose=True
)

## 3. 収束過程の可視化

In [ ]:
import matplotlib.pyplot as plt

iters = [entry['iter'] for entry in log]
rel_gaps = [entry['rel_gap'] for entry in log]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(iters, rel_gaps, 'o-', markersize=4)
ax.set_xlabel("Iteration")
ax.set_ylabel("Relative Duality Gap")
ax.set_title("iTAPAS Convergence (Sioux Falls Network)")
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-3, color='r', linestyle='--', label='Threshold (1e-3)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. 結果の確認

リンクフローと V/C 比 (Volume/Capacity ratio) を確認する。

In [ ]:
import pandas as pd
import numpy as np

# リンクフロー結果の表示
link_df = pd.read_csv(f"{data_dir}/link.csv")
results = pd.DataFrame({
    'link_id': link_df['link_id'].values,
    'from': link_df['from_node_id'].values,
    'to': link_df['to_node_id'].values,
    'flow': link_flow,
    'capacity': net.link_capacity,
    'V/C': link_flow / net.link_capacity,
    'fftt': net.link_fftt,
    'travel_time': bpr_cost(link_flow, net.link_fftt, net.link_capacity,
                            net.link_alpha, net.link_beta)
})

print(f"総リンクフロー: {link_flow.sum():.0f}")
print(f"最大 V/C 比:    {results['V/C'].max():.3f}")
print(f"平均 V/C 比:    {results['V/C'].mean():.3f}")
print()

# V/C比が高い上位10リンク
print("V/C比上位10リンク:")
results.sort_values('V/C', ascending=False).head(10)

In [ ]:
# V/C比の分布
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(results['V/C'], bins=20, edgecolor='black', alpha=0.7)
ax.set_xlabel("V/C Ratio")
ax.set_ylabel("Number of Links")
ax.set_title("Distribution of Volume/Capacity Ratio")
ax.axvline(x=1.0, color='r', linestyle='--', label='V/C = 1.0')
ax.legend()
plt.tight_layout()
plt.show()